
# CSE 5509 Final Project: Ego-Centered 360° Minimap from Single-Camera Views

This demo notebook runs the end-to-end pipeline for building an ego-centered qualitative minimap from single-camera images captured at fixed locations while rotating in place.

**Expected input layout**
- `PROJECT_DIR/bev_pipeline.py`
- `PROJECT_DIR/data/loc*/direction*.jpg`

**Main outputs**
- `outputs/per_location/loc*_minimap.png` (primary, object-level)
- `outputs/per_location/loc*_stitched_bev.png` (diagnostic, pixel-level)
- `outputs/tables/*.json|*.csv` and `outputs/run_report.json`


## 1) Setup / imports

In [ ]:

from pathlib import Path
import sys
import json
import importlib

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Edit this path for local or Colab usage.
PROJECT_DIR = Path('/content/drive/MyDrive/CSE 5509 Final Project' if IN_COLAB else '.').expanduser().resolve()

if not (PROJECT_DIR / 'bev_pipeline.py').exists():
    raise FileNotFoundError(f'bev_pipeline.py not found at: {PROJECT_DIR}')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import bev_pipeline
importlib.reload(bev_pipeline)
from bev_pipeline import PipelineConfig, resolve_project_paths

paths = resolve_project_paths(PROJECT_DIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('Data dir   :', paths['data_dir'])
print('Output dir :', paths['output_dir'])


## 2) Configuration

In [ ]:

CONFIG = PipelineConfig(
    repo_root=paths['repo_root'],
    data_dir=paths['data_dir'],
    output_dir=paths['output_dir'],
    run_small_demo=True,
    demo_locations=1,
    demo_images_per_location=None,  # None => use all images in selected location(s)
    detection_threshold=0.70,
    minimap_min_confidence=0.70,
    zero_shot_box_threshold=0.35,
    zero_shot_text_threshold=0.30,
    clean_output_dir=False,
)

# Set run_small_demo=False to process all locations.
print('detection_threshold      =', CONFIG.detection_threshold)
print('minimap_min_confidence   =', CONFIG.minimap_min_confidence)
print('zero_shot_box_threshold  =', CONFIG.zero_shot_box_threshold)
print('zero_shot_text_threshold =', CONFIG.zero_shot_text_threshold)


## 3) Dataset discovery

In [ ]:

from bev_pipeline import discover_dataset

summary = discover_dataset(CONFIG.data_dir)
print('Locations:', summary['location_count'])
print('Total images:', summary['total_images'])
print('Images per location:')
for loc, n in summary['images_per_location'].items():
    print(f'  - {loc}: {n}')

example_loc = summary['locations'][0]
print('
Example image paths:')
for p in summary['files'][example_loc][:3]:
    print(' ', p)


## 4) Load models

In [ ]:

from bev_pipeline import default_device, default_model_states

DEVICE = default_device()
model_states = default_model_states(CONFIG, device=DEVICE)
print('Device:', DEVICE)
print('segmentation:', model_states['seg'].get('available', False))
print('depth       :', model_states['depth'].get('available', False))
print('mask_rcnn   :', model_states['det'].get('available', False))
print('grounding_dino:', model_states['det_zero_shot'].get('available', False))

if not model_states['det_zero_shot'].get('available', False):
    print('Grounding DINO unavailable; pipeline will continue with Mask R-CNN only.')


## 5) Run pipeline

In [ ]:

from bev_pipeline import run_pipeline

run_report = run_pipeline(CONFIG, summary, model_states)
print('locations_processed:', run_report['locations_processed'])
print('images_processed   :', run_report['images_processed'])
print('total_detections   :', run_report['total_detections'])
print('output_dirs        :', json.dumps(run_report['output_dirs'], indent=2))


## 6) Display final results

In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

loc_result = run_report['location_results'][0]

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open(loc_result['minimap']))
ax[0].set_title('Final minimap (object-level)')
ax[0].axis('off')
ax[1].imshow(Image.open(loc_result['stitched_bev']))
ax[1].set_title('Stitched BEV (pixel-level diagnostic)')
ax[1].axis('off')
plt.show()

sample_img = loc_result['images'][0]
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open(sample_img['det_path']))
ax[0].set_title('Detection overlay')
ax[0].axis('off')
ax[1].imshow(Image.open(sample_img['bev_path']))
ax[1].set_title('Per-image BEV diagnostic')
ax[1].axis('off')
plt.show()


## 7) Inspect tables

In [ ]:

import pandas as pd
from bev_pipeline import summarize_detection_table

table_path = Path(loc_result['minimap_instance_table_csv'])
df = pd.read_csv(table_path)
cols = [
    'source_image','class_name','instance_label','confidence','detector_source',
    'estimated_depth_m','bearing_deg','range_m','minimap_xy'
]
display(df[cols].head(15))

rows = pd.read_json(Path(loc_result['minimap_instance_table_json'])).to_dict(orient='records')
stats = summarize_detection_table(rows)
print('final detection count:', stats['final_detection_count'])
print('detections per class:', stats['detections_per_class'])
print('detections per direction:', stats['detections_per_direction'])


## 8) Diagnostics and limitations

In [ ]:

dedup_diag_path = Path(CONFIG.output_dir) / 'tables' / f"{loc_result['location']}_dedup_diagnostics.json"
stitch_diag_path = Path(CONFIG.output_dir) / 'tables' / f"{loc_result['location']}_stitch_diagnostics.json"

dedup_diag = json.loads(dedup_diag_path.read_text()) if dedup_diag_path.exists() else []
stitch_diag = json.loads(stitch_diag_path.read_text()) if stitch_diag_path.exists() else []

print('Dedup suppressed count:', len(dedup_diag))
print('Stitch diagnostics rows:', len(stitch_diag))
print('
Common failure modes:')
print('- Monocular depth is approximate and not calibrated metric depth.')
print('- Detector false positives and missed objects are expected.')
print('- Adjacent directions can produce repeated detections.')
print('- FOV is approximated unless camera intrinsics are calibrated.')
print('- BBox bottom-center is a proxy for ground contact.')



## 9) AI usage statement
AI tools were used for code cleanup, docstring drafting, and notebook organization. Final project claims, settings, and limitations were reviewed and kept aligned with a qualitative course-project scope.
